# Baselines 1 and 2

**Baseline 1** — the detector with no abstention. FROC, sensitivity, false-negative rate.
**Baseline 2** — confidence-threshold abstention. Sensitivity against abstention rate.

Both are inference on the frozen detector. No training.

## Written against four failures from the last attempt

**A 12-hour Colab cap killed the run and everything in `/content` went with it.** Per-image
detection JSONs now sync to Drive every 50 images, so a session death costs 50 images
rather than the lot. Re-running resumes from what Drive already has.

**Reading 4,882 `.mha` files over the mounted Drive stalled inside a C call** where
`KeyboardInterrupt` cannot reach it — it looks like an infinite loop and needs a runtime
restart. Source files are bulk-copied to local disk first.

**Scoring everything in one pass does not fit in a session.** Positives run first (1,134
images, ~25 min). The 3,748 clean images are a second pass, needed only for a true
false-positive rate.

**An empty result printed nothing and looked like success.** The table-building cell now
asserts rather than producing an empty DataFrame three cells before anyone notices.

## 1 · Mount, paths, and a checkpoint helper

In [ ]:
# ===========================================================================
# CANONICAL DRIVE PATHS
# Mirrored from src/paths.py and notebooks/CONFIG_CELL.md. Mapped from Drive
# 2026-09-12 with the Drive connector. Change all three together.
# Full layout, folder ids and the old->new table: DRIVE_LAYOUT.md
# ===========================================================================
!pip -q install SimpleITK
import os, json, glob, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

# --- mount -----------------------------------------------------------------
# ismount(), not isdir(). A plain local directory under an unmounted
# /content/drive is also a dir, and creating one blocks the mount and makes
# Drive look empty -- that happened once and looked like a wiped Drive.
if not os.path.ismount('/content/drive'):
    if Path('/content/drive').exists():
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'mount failed'

# --- resolve the root ------------------------------------------------------
# MyDrive/Algoverse is a SHORTCUT to the shared folder Feliciano_Algoverse.
# One folder, not two; the FUSE mount resolves it as a directory.
# Test for the 01_data marker, never for the root itself: an unresolved
# shortcut and a stale empty directory both "exist", and that is exactly how
# a stray empty results/ tree got created on 2026-09-12.
ROOT = None
for _c in ['/content/drive/MyDrive/Algoverse',
           '/content/drive/MyDrive/Feliciano_Algoverse',
           '/content/drive/Shareddrives/Feliciano_Algoverse']:
    if (Path(_c)/'01_data').is_dir():
        ROOT = Path(_c); break
assert ROOT is not None, (
    'Algoverse root not found. Tried MyDrive/Algoverse, '
    'MyDrive/Feliciano_Algoverse, Shareddrives/Feliciano_Algoverse.\n'
    f'MyDrive top level: '
    f'{sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20]}\n'
    'MyDrive/Algoverse is a shortcut to the shared Feliciano_Algoverse. If it '
    'is gone: Drive -> Shared with me -> right-click Feliciano_Algoverse -> '
    'Add shortcut to Drive -> My Drive.')

# --- layout ----------------------------------------------------------------
SOURCE    = ROOT/'01_data'/'00_source'           # node21, chexpert, mimic_cxr
NODE21    = SOURCE/'node21'
MHA_SRC   = NODE21/'images'                      # 4,882 .mha
ANN_CSV   = NODE21/'metadata.csv'                # 5,224 rows, 1,476 label==1
GRID      = ROOT/'01_data'/'01_grid'
GRID_CSV  = GRID/'grid_v5.csv'                   # 231,145 bytes if it is the right one
RUNS      = GRID/'_runs'                         # generation checkpoint zips, not data
EMB_DIR   = ROOT/'01_data'/'02_embeddings'
CPASTE    = ROOT/'01_data'/'03_copypaste'
MODELS    = ROOT/'02_results'/'00_models'
CKPT      = MODELS/'baseline1_checkpoint.pth'    # .pth -- the old .pt path is dead
FIGS      = ROOT/'02_results'/'01_figures'
B1_DIR    = ROOT/'02_results'/'02_baselines'
PRED_DIR  = ROOT/'02_results'/'03_predictor'
B3_DIR    = ROOT/'02_results'/'04_baseline3'

print(f'root: {ROOT}')

OUT  = Path('/content/baselines');      (OUT/'dets').mkdir(parents=True, exist_ok=True)
DEST = B1_DIR;                          (DEST/'dets').mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

# --- verify inputs ---------------------------------------------------------
for _p in (MHA_SRC, ANN_CSV, CKPT):
    assert _p.exists(), (
        f'{_p} missing.\n  its parent holds: '
        f'{sorted(q.name for q in _p.parent.iterdir())[:12] if _p.parent.is_dir() else "parent does not exist either"}'
        '\n  see the old->new table in DRIVE_LAYOUT.md')
print('inputs ok')

# --- stray folders from the pre-reorganisation layout ----------------------
_stray = [ROOT/'results', ROOT/'data', ROOT/'artifacts', ROOT/'grid_v5b_runs',
          Path('/content/drive/MyDrive/grid_v5_runs'),
          Path('/content/drive/MyDrive/grid_v4_runs'),
          Path('/content/drive/MyDrive/Teammates')]
_hits = [(p, len(list(p.iterdir()))) for p in _stray if p.is_dir()]
if _hits:
    print('\nstray folders from the old layout (DRIVE_LAYOUT.md lists what to do):')
    for p, n in _hits:
        print(f'  {p}   ({"empty, safe to delete" if n == 0 else str(n) + " items -- CHECK"})')


def sync_to_drive(sub='dets'):
    """Copy anything new to Drive. Cheap, idempotent, and the only reason a dead session
    is survivable -- /content does not persist."""
    n = 0
    for f in (OUT/sub).iterdir():
        if not (DEST/sub/f.name).exists():
            shutil.copy(f, DEST/sub/f.name); n += 1
    return n


def restore_from_drive(sub='dets'):
    """Pull back whatever a previous session managed to save."""
    n = 0
    for f in (DEST/sub).iterdir():
        if not (OUT/sub/f.name).exists():
            shutil.copy(f, OUT/sub/f.name); n += 1
    return n

print(f'\nrestored {restore_from_drive()} detection files from a previous session')
print(f'writing to {DEST}')


## 2 · Stage source images locally

Only the annotated images to begin with. `INCLUDE_NEGATIVES` adds the 3,748 clean ones,
which triples the runtime — leave it off for the first pass.

In [ ]:
import pandas as pd

INCLUDE_NEGATIVES = False        # second pass; needed only for a true FP rate

raw = pd.read_csv(ANN_CSV)
raw = raw[raw.img_name != 'n0507.mha']      # byte-identical duplicate of n1059, and the
                                            # less complete of the two annotations
gt  = {n: g for n, g in raw[raw.label == 1].groupby('img_name')}
names = sorted(gt)
if INCLUDE_NEGATIVES:
    names += sorted(set(raw[raw.label == 0].img_name) - set(gt))
print(f'{len(names)} images to score ({len(gt)} with nodules)')

LOCAL = Path('/content/node21'); LOCAL.mkdir(exist_ok=True)
import subprocess
t0, staged = time.time(), 0
for i, n in enumerate(names):
    if not (LOCAL/n).exists():
        subprocess.run(['cp', str(MHA_SRC/n), str(LOCAL/n)], check=True); staged += 1
    if i % 200 == 0:
        print(f'  {i}/{len(names)}  ({time.time()-t0:.0f}s)')
MHA_DIR = LOCAL
print(f'staged {staged} new files in {time.time()-t0:.0f}s, '
      f'{len(list(LOCAL.glob("*.mha")))} present, '
      f'{sum(f.stat().st_size for f in LOCAL.glob("*.mha"))/1e9:.1f} GB')

## 3 · Detector and preprocessing

In [ ]:
import numpy as np, cv2, SimpleITK as sitk, torch, torchvision
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF

SIZE, DET_SIZE, SCORE_MIN, CHEST_MM = 512, 800, 0.05, 350.0
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

DET = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=None, weights_backbone=None,
    box_score_thresh=0.0,          # F6: two stacked filters at 0.05 truncated the old
    box_detections_per_img=300)    # FROC curve into five duplicate operating points
DET.roi_heads.box_predictor = FastRCNNPredictor(
    DET.roi_heads.box_predictor.cls_score.in_features, 2)
_st = torch.load(CKPT, map_location=DEV, weights_only=False)
DET.load_state_dict(_st['model'] if isinstance(_st, dict) and 'model' in _st else _st)
DET = DET.eval().to(DEV)
print(f'detector on {DEV}; internal transform min_size={DET.transform.min_size}')


def load_chest(path, size=SIZE):
    """IDENTICAL to the generation pipeline. If this drifts from the synthetic path, any
    real-vs-synthetic comparison confounds resolution with lesion type."""
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    oh, ow = a.shape
    s = min(oh, ow); x0, y0 = (ow-s)//2, (oh-s)//2
    a = a[y0:y0+s, x0:x0+s]
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    a = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((a*255).astype(np.uint8)).astype(np.float32)/255.0
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    return Image.fromarray((a*255).astype(np.uint8)).convert('RGB'), (ow, oh, s, x0, y0)


@torch.no_grad()
def detect_all(pil, size=DET_SIZE):
    im = pil.convert('RGB').resize((size,size), Image.LANCZOS)
    o = DET([TF.to_tensor(im).to(DEV)])[0]
    return o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()


def centre_in(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]

## 4 · Score — resumable, checkpointed

Safe to interrupt and safe to re-run. Every image writes one JSON containing **all** boxes
above 0.05, so no later threshold change or matching-rule sweep needs inference again.

In [ ]:
SYNC_EVERY = 50

done = {p.stem for p in (OUT/'dets').glob('*.json')}
todo = [n for n in names if n.replace('.mha','') not in done]
print(f'{len(done)} already scored, {len(todo)} to go')

t0 = time.time()
for i, name in enumerate(todo):
    p = MHA_DIR/name
    if not p.exists():
        continue
    img, (W,H,s,x0,y0) = load_chest(p)
    b, sc = detect_all(img)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist(),
               'crop': [int(W), int(H), int(s), int(x0), int(y0)]},
              open(OUT/'dets'/f'{name.replace(".mha","")}.json', 'w'))

    if (i+1) % SYNC_EVERY == 0 or i == len(todo)-1:
        n = sync_to_drive()
        el = time.time()-t0
        rate = el/(i+1)
        print(f'  {i+1}/{len(todo)}  {el/60:5.1f} min  '
              f'~{rate*(len(todo)-i-1)/60:5.0f} min left  +{n} to Drive')

print(f'\ndone. {len(list((OUT/"dets").glob("*.json")))} detection files, '
      f'{sync_to_drive()} final sync')

## 5 · Build the tables

Reads only the JSONs, so it works even if scoring was interrupted. The assertion is
deliberate: the previous version produced an empty DataFrame and printed nothing, which
looked like success and was not noticed for three cells.

In [ ]:
rows, nod, dropped = [], [], []
for stem in sorted({p.stem for p in (OUT/'dets').glob('*.json')}):
    d = json.load(open(OUT/'dets'/f'{stem}.json'))
    b, sc = np.array(d['boxes']).reshape(-1,4), np.array(d['scores'])
    W, H, s, x0, y0 = d['crop']
    g = gt.get(f'{stem}.mha')
    n_gt = 0
    if g is not None:
        for j, r in enumerate(g.itertuples()):
            fx0, fy0 = (r.x-x0)/s, (r.y-y0)/s
            fx1, fy1 = (r.x+r.width-x0)/s, (r.y+r.height-y0)/s
            if not (0 <= fx0 < fx1 <= 1 and 0 <= fy0 < fy1 <= 1):
                dropped.append((stem, 'box outside crop')); continue   # drop, never clamp
            t = (fx0, fy0, fx1, fy1); n_gt += 1
            hits = [ss for bb, ss in zip(b, sc) if centre_in(bb, t)]
            nod.append(dict(img_name=stem, nodule=j,
                            fx0=fx0, fy0=fy0, fx1=fx1, fy1=fy1,
                            approx_mm=round(CHEST_MM*(fx1-fx0), 1),
                            best_score=round(float(max(hits, default=0.0)), 4)))
    rows.append(dict(img_name=stem, n_gt=n_gt, n_boxes=int(len(sc)),
                     max_score=round(float(sc.max()) if len(sc) else 0.0, 4)))

img_df = pd.DataFrame(rows); nod_df = pd.DataFrame(nod)
assert len(img_df) > 0,  'no detection files found -- did the scoring cell run?'
assert len(nod_df) > 0, (f'no nodules scored from {len(img_df)} images. '
                         f'{len(dropped)} boxes fell outside the crop -- the coordinate '
                         f'transform is wrong if that is most of them.')

for df, name in [(img_df,'per_image.csv'), (nod_df,'per_nodule.csv')]:
    df.to_csv(OUT/name, index=False); shutil.copy(OUT/name, DEST/name)
pd.DataFrame(dropped, columns=['img_name','reason']).to_csv(OUT/'dropped.csv', index=False)

print(f'{len(img_df)} images, {len(nod_df)} nodules, {len(dropped)} boxes dropped')
print(nod_df.best_score.describe().round(3).to_string())

### Look before believing

If the green boxes are not on visible nodules, the coordinate transform is wrong and every
number after this is meaningless. That is exactly how the synthetic grid was invalidated
the first time, in a different pipeline.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
k = 6
fig, ax = plt.subplots(1, k, figsize=(2.9*k, 3.4))
for a_, (_, r) in zip(ax, nod_df.sample(k, random_state=0).iterrows()):
    img, _ = load_chest(MHA_DIR/f'{r.img_name}.mha')
    a_.imshow(img, cmap='gray', extent=[0,1,1,0])
    a_.add_patch(patches.Rectangle((r.fx0, r.fy0), r.fx1-r.fx0, r.fy1-r.fy0,
                 fill=False, edgecolor='lime', lw=2))
    a_.set_title(f'{r.img_name}\ndet {r.best_score:.3f}  {r.approx_mm:.0f}mm', fontsize=8)
    a_.set_xlim(0,1); a_.set_ylim(1,0); a_.axis('off')
plt.suptitle('green must sit on an actual nodule', fontsize=10)
plt.tight_layout(); plt.show()

## 6 · Baseline 1 — FROC

Sensitivity against false positives per image, swept over score thresholds.

> With `INCLUDE_NEGATIVES = False` the false-positive rate is computed over annotated
> images only, so it is **not** the true FP rate — those images contain nodules the
> detector is meant to find. Report it as such, or run the second pass.

In [ ]:
def froc(nod_df, img_df):
    thresholds = np.unique(np.concatenate([np.linspace(0, 1, 201),
                                           nod_df.best_score.values]))
    # cache each image's boxes and targets once rather than re-reading per threshold
    cache = []
    for _, r in img_df.iterrows():
        d = json.load(open(OUT/'dets'/f'{r.img_name}.json'))
        b, sc = np.array(d['boxes']).reshape(-1,4), np.array(d['scores'])
        g = nod_df[nod_df.img_name == r.img_name]
        cache.append((b, sc, [(x.fx0, x.fy0, x.fx1, x.fy1) for x in g.itertuples()]))

    n_img, n_nod, pts = len(img_df), len(nod_df), []
    for t in thresholds:
        tp = int((nod_df.best_score >= t).sum())
        fp = sum(sum(1 for bb in b[sc >= t] if not any(centre_in(bb, g) for g in tg))
                 for b, sc, tg in cache)
        pts.append((t, tp/n_nod, fp/n_img))
    return pd.DataFrame(pts, columns=['threshold','sensitivity','fp_per_image'])

F = froc(nod_df, img_df)
F.to_csv(OUT/'froc.csv', index=False); shutil.copy(OUT/'froc.csv', DEST)

OPS = [0.125, 0.25, 0.5, 1, 2, 4, 8]
T = pd.DataFrame([dict(fp_per_image=op,
                       sensitivity=round(F[F.fp_per_image <= op].sensitivity.max(), 4)
                       if (F.fp_per_image <= op).any() else np.nan) for op in OPS])
print(T.to_string(index=False))
print(f'\nFROC score (mean of 7 operating points): {T.sensitivity.mean():.4f}')
print(f'distinct sensitivity values: {T.sensitivity.nunique()} of 7')
if T.sensitivity.nunique() < 5:
    print('  -> duplicates. Either the detector runs out of boxes above 0.05, or a '
          'threshold is truncating the curve (F6).')
T.to_csv(OUT/'table-froc.csv', index=False); shutil.copy(OUT/'table-froc.csv', DEST)

## 7 · Baseline 2 — confidence-threshold abstention

Abstain on an image when the detector's top score anywhere is below a threshold, then
measure the false-negative rate on what remains.

Every rate carries a Wilson interval. Wilson rather than normal because these sit near 0
and 1, where the normal interval misbehaves — and because Table 1 currently has no
intervals at all.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p, d = k/n, 1 + z**2/n
    c = (p + z**2/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))/d
    return max(0, c-h), min(1, c+h)

merged = nod_df.merge(img_df[['img_name','max_score']], on='img_name')
assert len(merged) > 0, ('merge produced nothing -- img_name mismatch: '
                         f'{sorted(set(nod_df.img_name))[:2]} vs '
                         f'{sorted(set(img_df.img_name))[:2]}')

rows = []
for a in np.arange(0.0, 0.95, 0.05):
    ans = merged[merged.max_score >= a]
    if len(ans) == 0: continue
    fn = int((ans.best_score < SCORE_MIN).sum())
    lo, hi = wilson(fn, len(ans))
    rows.append(dict(abstain_below=round(a,2),
                     abstention_rate=round(1-len(ans)/len(merged), 4),
                     n_answered=len(ans), fnr_answered=round(fn/len(ans), 4),
                     fnr_lo=round(lo,4), fnr_hi=round(hi,4)))
B2 = pd.DataFrame(rows)
assert len(B2) > 0, 'abstention sweep produced no rows'
B2.to_csv(OUT/'table-baseline2.csv', index=False)
shutil.copy(OUT/'table-baseline2.csv', DEST)
print(B2.to_string(index=False))

## 8 · Figure and final sync

In [ ]:
INK,EDGE,GRIDC,BLUE,DEEP,SLATE = '#2c3e50','#1f2d4d','#eeeeee','#6694de','#3c5182','#7982a6'
plt.rcParams.update({'savefig.dpi':300,'figure.facecolor':'white','font.size':12,
    'axes.titleweight':'bold','axes.titlecolor':INK,'axes.labelweight':'bold',
    'axes.labelcolor':INK,'text.color':INK,'xtick.color':INK,'ytick.color':INK,
    'axes.edgecolor':INK,'axes.spines.top':False,'axes.spines.right':False,
    'grid.color':GRIDC,'axes.axisbelow':True})

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
f = F[F.fp_per_image > 0].sort_values('fp_per_image')
ax[0].semilogx(f.fp_per_image, f.sensitivity, color=BLUE, lw=2.4)
ax[0].scatter(T.fp_per_image, T.sensitivity, color=DEEP, s=42, zorder=5, edgecolor=EDGE)
ax[0].set_xlabel('False positives per image'); ax[0].set_ylabel('Sensitivity')
ax[0].set_title(f'Baseline 1 — FROC, mean {T.sensitivity.mean():.3f}')
ax[0].set_ylim(0, 1.02); ax[0].grid(True, which='both', color=GRIDC)

ax[1].plot(B2.abstention_rate, B2.fnr_answered, color=SLATE, lw=2.4, marker='o', ms=4)
ax[1].fill_between(B2.abstention_rate, B2.fnr_lo, B2.fnr_hi, color=SLATE, alpha=.2)
ax[1].set_xlabel('Abstention rate'); ax[1].set_ylabel('FNR on answered cases')
ax[1].set_title('Baseline 2 — confidence-only abstention')
ax[1].grid(True, color=GRIDC)
plt.suptitle('Baselines on NODE21', fontsize=17, fontweight='bold', color=INK)
plt.tight_layout(rect=[0,0,1,0.93])
for ext in ('png','pdf'):
    fig.savefig(OUT/f'fig-baselines.{ext}', dpi=300, bbox_inches='tight', facecolor='white')
    shutil.copy(OUT/f'fig-baselines.{ext}', DEST)
    shutil.copy(OUT/f'fig-baselines.{ext}', FIGS)
plt.show()

print(f'\nsynced {sync_to_drive()} files; everything also at {DEST}')
print('\nNOTE: without the original train/val split, these pool training and validation.')
print('The per-image CSVs are saved, so re-slicing when the split turns up costs nothing.')

---

# 9 · Rescoring with the preprocessing the detector was actually trained on

Everything above fed `baseline1_checkpoint.pth` images produced by `load_chest`:
centre-crop to square, 1/99 percentile clip, CLAHE, resize to 512, fractional box
coordinates.

`algoverse_dataset.py` — recovered 2026-09-12, the code that actually trained this
checkpoint — does none of that:

```python
arr = sitk.GetArrayFromImage(sitk.ReadImage(path)).squeeze()
arr = arr.astype(np.float32) / max(arr.max(), 1.0)   # max-normalise, nothing else
img = Image.fromarray(arr, mode='F')                 # ToTensor, repeat to 3 channels
# boxes: [x, y, x+w, y+h] in NATIVE pixels
```

No CLAHE, no clipping, no crop, no resize — the model's own `GeneralizedRCNNTransform`
does the scaling. So §6's FROC of 0.6717 is this detector evaluated **out of
distribution**, which is the most plausible reason it sits so far below the 0.885/0.889
in the earlier draft.

This section scores the same 1,133 images the training-matched way and reports the
delta. Two further things it fixes:

**No boxes can be dropped.** `load_chest` crops, and §5 *discards* ground-truth boxes
that fall outside the crop rather than clamping. None were dropped on the current run,
but that was luck. Native coordinates need no transform, so the failure mode does not
exist.

**Train and val are finally separable.** `splits.csv` is recovered. The checkpoint
trained on the train split, so **val-only FROC is the honest number** and everything
above pooled train and val together.

Keep both passes. The grid is intrinsically 512+CLAHE because RadEdit only runs at 512,
so any synthetic-versus-real comparison needs the 512 numbers for parity — while
absolute capability claims need these. Reporting both turns a confound into a measured
quantity.

In [ ]:
# --- the recovered split -----------------------------------------------------
SPLIT_CSV = None
for _c in [SOURCE/'baseline1_splits.csv', SOURCE/'splits.csv',
           NODE21/'baseline1_splits.csv', NODE21/'splits.csv']:
    if _c.exists():
        SPLIT_CSV = _c; break
if SPLIT_CSV is None:
    _hits = sorted(glob.glob('/content/drive/MyDrive/**/baseline1_splits.csv', recursive=True)
                   + glob.glob('/content/drive/MyDrive/**/splits.csv', recursive=True))
    _hits += sorted(glob.glob('/content/*splits.csv'))
    assert _hits, (
        'splits.csv not found. It is in Maia\'s Drive folder and in Dhruv\'s\n'
        'Algoverse_Predictor_Code_Outputs/data/splits/baseline1_splits.csv (77,012 bytes).\n'
        f'Copy it to {SOURCE}/baseline1_splits.csv, or drop it in /content.')
    SPLIT_CSV = Path(_hits[0])
    print(f'found by glob: {SPLIT_CSV}')

SPL = pd.read_csv(SPLIT_CSV)
assert {'img_name', 'split'} <= set(SPL.columns), f'unexpected columns: {list(SPL.columns)}'
print(f'{SPLIT_CSV}  ({SPLIT_CSV.stat().st_size:,} bytes)')
if SPLIT_CSV.stat().st_size != 77012:
    print('  WARNING: not 77,012 bytes -- may not be the split the checkpoint used')
print(SPL.split.value_counts().to_string())
SPLIT_OF = dict(zip(SPL.img_name, SPL.split))
print(f'\ncoverage of our {len(names)} scored images: '
      f'{sum(n in SPLIT_OF for n in names)}')

In [ ]:
# --- training-matched preprocessing and inference ----------------------------
OUT_N  = OUT/'dets_native';   OUT_N.mkdir(parents=True, exist_ok=True)
DEST_N = DEST/'dets_native';  DEST_N.mkdir(parents=True, exist_ok=True)


def load_native(path):
    """Byte-for-byte the preprocessing in algoverse_dataset.NoduleDataset."""
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    a = np.asarray(a).squeeze()
    if a.ndim != 2:
        raise ValueError(f'expected 2D after squeeze, got {a.shape}')
    m = float(a.max()) if a.max() > 0 else 1.0
    return a.astype(np.float32)/m


@torch.no_grad()
def detect_native(a):
    """No resize. The model's GeneralizedRCNNTransform scales internally, exactly as it
    did in training. Boxes come back in NATIVE pixel coordinates."""
    t = torch.from_numpy(a)[None].repeat(3, 1, 1)
    o = DET([t.to(DEV)])[0]
    return o['boxes'].cpu().numpy(), o['scores'].cpu().numpy()


def sync_native():
    n = 0
    for f in OUT_N.iterdir():
        if not (DEST_N/f.name).exists():
            shutil.copy(f, DEST_N/f.name); n += 1
    return n


for f in DEST_N.iterdir():            # restore a previous session
    if not (OUT_N/f.name).exists():
        shutil.copy(f, OUT_N/f.name)

done = {p.stem for p in OUT_N.glob('*.json')}
todo = [n for n in names if n.replace('.mha', '') not in done]
print(f'{len(done)} already scored, {len(todo)} to go')

t0 = time.time()
for i, name in enumerate(todo):
    p = MHA_DIR/name
    assert p.exists(), f'{p} missing -- staging is incomplete, do not let this skip'
    a = load_native(p)
    b, sc = detect_native(a)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist(),
               'shape': [int(a.shape[0]), int(a.shape[1])]},
              open(OUT_N/f'{name.replace(".mha", "")}.json', 'w'))
    if (i+1) % 50 == 0 or i == len(todo)-1:
        el = time.time()-t0
        print(f'  {i+1}/{len(todo)}  {el/60:5.1f} min  '
              f'~{el/(i+1)*(len(todo)-i-1)/60:5.0f} min left  +{sync_native()} to Drive')
print(f'\n{len(list(OUT_N.glob("*.json")))} native detection files, {sync_native()} synced')

In [ ]:
# --- tables, with NO coordinate transform -----------------------------------
def centre_in_px(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]


rows, nod = [], []
for stem in sorted({p.stem for p in OUT_N.glob('*.json')}):
    d = json.load(open(OUT_N/f'{stem}.json'))
    b, sc = np.array(d['boxes']).reshape(-1, 4), np.array(d['scores'])
    H, W = d['shape']
    g = gt.get(f'{stem}.mha')
    n_gt = 0
    if g is not None:
        for j, r in enumerate(g.itertuples()):
            t = (r.x, r.y, r.x+r.width, r.y+r.height)   # native pixels, as trained
            n_gt += 1
            hits = [ss for bb, ss in zip(b, sc) if centre_in_px(bb, t)]
            nod.append(dict(img_name=stem, nodule=j, split=SPLIT_OF.get(f'{stem}.mha', 'unknown'),
                            x0=t[0], y0=t[1], x1=t[2], y1=t[3],
                            side_px=max(r.width, r.height),
                            best_score=round(float(max(hits, default=0.0)), 4)))
    rows.append(dict(img_name=stem, n_gt=n_gt, n_boxes=int(len(sc)),
                     split=SPLIT_OF.get(f'{stem}.mha', 'unknown'),
                     max_score=round(float(sc.max()) if len(sc) else 0.0, 4)))

img_n = pd.DataFrame(rows); nod_n = pd.DataFrame(nod)
assert len(img_n) and len(nod_n), 'no native detections -- did the cell above run?'
print(f'{len(img_n)} images, {len(nod_n)} nodules -- NONE dropped (no crop, no transform)')
print(f'\n512+CLAHE pass had {len(nod_df)} nodules; native has {len(nod_n)}')
print(nod_n.split.value_counts().to_string())
img_n.to_csv(OUT/'per_image_native.csv', index=False); shutil.copy(OUT/'per_image_native.csv', DEST)
nod_n.to_csv(OUT/'per_nodule_native.csv', index=False); shutil.copy(OUT/'per_nodule_native.csv', DEST)

In [ ]:
# --- FROC, both pipelines, split-aware --------------------------------------
def froc_px(nd, im, detdir, native):
    ths = np.unique(np.concatenate([np.linspace(0, 1, 201), nd.best_score.values]))
    cache = []
    for _, r in im.iterrows():
        d = json.load(open(detdir/f'{r.img_name}.json'))
        b, sc = np.array(d['boxes']).reshape(-1, 4), np.array(d['scores'])
        g = nd[nd.img_name == r.img_name]
        tg = ([(x.x0, x.y0, x.x1, x.y1) for x in g.itertuples()] if native
              else [(x.fx0, x.fy0, x.fx1, x.fy1) for x in g.itertuples()])
        cache.append((b, sc, tg))
    n_img, n_nod, pts = len(im), len(nd), []
    for t in ths:
        tp = int((nd.best_score >= t).sum())
        fp = sum(sum(1 for bb in b[sc >= t] if not any(centre_in_px(bb, q) for q in tg))
                 for b, sc, tg in cache)
        pts.append((t, tp/n_nod, fp/n_img))
    return pd.DataFrame(pts, columns=['threshold', 'sensitivity', 'fp_per_image'])


OPS = [0.125, 0.25, 0.5, 1, 2, 4, 8]


def ops_table(F):
    return pd.DataFrame([dict(fp_per_image=op,
        sensitivity=round(F[F.fp_per_image <= op].sensitivity.max(), 4)
        if (F.fp_per_image <= op).any() else np.nan) for op in OPS])


cmp_rows = []
for tag, nd, im, dd, nat in [
        ('512+CLAHE (as trained? NO)', nod_df, img_df, OUT/'dets',        False),
        ('native (as trained)',        nod_n,  img_n,  OUT/'dets_native', True)]:
    for scope in ['all', 'val']:
        if scope == 'val':
            if 'split' not in nd.columns:
                continue
            keep = set(nd[nd.split == 'val'].img_name)
            nd2, im2 = nd[nd.img_name.isin(keep)], im[im.img_name.isin(keep)]
            if len(nd2) < 20:
                print(f'  skipping {tag} / val -- only {len(nd2)} nodules'); continue
        else:
            nd2, im2 = nd, im
        T2 = ops_table(froc_px(nd2, im2, dd, nat))
        cmp_rows.append(dict(pipeline=tag, scope=scope, n_img=len(im2), n_nod=len(nd2),
                             froc_mean=round(T2.sensitivity.mean(), 4),
                             **{f'sens@{o}': T2.sensitivity.iloc[k]
                                for k, o in enumerate(OPS)}))
        print(f'  {tag:28} {scope:4} n={len(im2):5} FROC {T2.sensitivity.mean():.4f}')

CMP = pd.DataFrame(cmp_rows)
CMP.to_csv(OUT/'table-preprocessing-comparison.csv', index=False)
shutil.copy(OUT/'table-preprocessing-comparison.csv', DEST)
print()
print(CMP.to_string(index=False))

print('\nHOW TO READ IT:')
print('  native >> 512+CLAHE  -> the 512 pipeline was costing real sensitivity and every')
print('     number in sections 5-8 above is an out-of-distribution measurement. Report the')
print('     native row as Baseline 1 and keep the 512 row only for grid parity.')
print('  the two are close       -> preprocessing was not the problem, and the gap to the')
print('     0.885/0.889 in the earlier draft is something else -- most likely that those')
print('     were val-only while sections 5-8 pooled train and val.')
print(f'\nsynced {sync_native()} native detections; tables also at {DEST}')